# 0. Setup

In [1]:
import ibis
import pandas as pd
from utils.f_0_dirs import get_data_dirs
from linearmodels.panel.results import PanelEffectsResults
from f_7_run_panel import run_panel, ModelSpec, format_str

dirs = get_data_dirs(segment="model")
con = ibis.duckdb.connect(dirs.db_path, read_only=True)

IOException: IO Error: Cannot open file "C:\Users\lazyst\Files\ucl\Dissertation\build\output\fame_data.duckdb": The process cannot access the file because it is being used by another process.

File is already open in 
C:\Users\lazyst\AppData\Local\python\pythoncore-3.14-64\python.exe (PID 22752)

# 0. 2-factor productivity

In [ ]:
out_file_name = "results_0_tfp"

table_panel_name = "working_yearly"
table_panel_old = "fame_yearly_kp"

t_panel = con.table(table_panel_name)
t_old = con.table(table_panel_old)
df_panel = (
    t_panel
    .drop('gva1_per_worker', 'gva2_per_worker')
    .distinct(on=['registered_number', 'year'])
    .left_join(
        t_old.select('registered_number', 'year', 'tangibles', 'intangibles', 'investments_other').distinct(on=['registered_number', 'year']),
        ['registered_number', 'year']
    )
    .execute()
)

models = {
    'gva1_ft': {'Y': 'gva1', 'X': ['total_assets', 'employees'], 'to_log': ['Y', 'X'], 'fe': ['i', 't']},
    'gva1_f': {'Y': 'gva1', 'X': ['total_assets', 'employees'], 'to_log': ['Y', 'X'], 'fe': ['i']},
    'gva1_t': {'Y': 'gva1', 'X': ['total_assets', 'employees'], 'to_log': ['Y', 'X'], 'fe': ['t']},
    'gva1_n': {'Y': 'gva1', 'X': ['total_assets', 'employees'], 'to_log': ['Y', 'X'], 'fe': []}
}

run_res_series: list[tuple[PanelEffectsResults, ModelSpec, str]] = []
worker_args = [
    (ModelSpec(**mod_obj), df_panel, m_name)
    for m_name, mod_obj in models.items() if ModelSpec(**mod_obj).include
]
for args in worker_args:
    mod, table_panel_name, model_name = args
    res, beta, effects_dict = run_panel(args)
    if res is None:
        print(f"Model '{model_name}' failed. Skipping.")
        continue
    run_res_series.append((res, mod, model_name))

model_count = len(run_res_series)
if model_count:
    output_str = "\n".join(map(format_str, run_res_series))
    print(f"Panel regressions complete. {model_count} model{'' if model_count == 1 else 's'}, writing.")
    with open(dirs.output_dir / f"{out_file_name}.txt", "w") as f:
        f.write(output_str)

Running model 'gva1_ft'
✅ Model 'gva1_ft' estimated: ln_total_assets=0.286, ln_employees=0.617
Running model 'gva1_f'
✅ Model 'gva1_f' estimated: ln_total_assets=0.284, ln_employees=0.613
Running model 'gva1_t'
✅ Model 'gva1_t' estimated: ln_total_assets=0.421, ln_employees=0.562
Running model 'gva1_n'
✅ Model 'gva1_n' estimated: ln_total_assets=0.421, ln_employees=0.561
Panel regressions complete. 4 models, writing.


In [ ]:
out_file_name = "results_0_tfp_alt_measure"

models = {
    'gva1': {'Y': 'gva1', 'X': ['total_assets', 'employees'], 'to_log': ['Y', 'X'], 'fe': ['i', 't']},
    'gva2': {'Y': 'gva2', 'X': ['total_assets', 'employees'], 'to_log': ['Y', 'X'], 'fe': ['i', 't']},
    'k1': {'Y': 'gva1', 'X': ['fixed_total', 'employees'], 'to_log': ['Y', 'X'], 'fe': ['i', 't']},
    'k2': {'Y': 'gva1', 'X': ['tangibles', 'intangibles', 'investments_other', 'employees'], 'to_log': ['Y', 'X'], 'fe': ['i', 't']}
}

run_res_series: list[tuple[PanelEffectsResults, ModelSpec, str]] = []
worker_args = [
    (ModelSpec(**mod_obj), df_panel, m_name)
    for m_name, mod_obj in models.items() if ModelSpec(**mod_obj).include
]
for args in worker_args:
    mod, table_panel_name, model_name = args
    res, beta, effects_dict = run_panel(args)
    if res is None:
        print(f"Model '{model_name}' failed. Skipping.")
        continue
    run_res_series.append((res, mod, model_name))

model_count = len(run_res_series)
if model_count:
    output_str = "\n".join(map(format_str, run_res_series))
    print(f"Panel regressions complete. {model_count} model{'' if model_count == 1 else 's'}, writing.")
    with open(dirs.output_dir / f"{out_file_name}.txt", "w") as f:
        f.write(output_str)

Running model 'gva1'
✅ Model 'gva1' estimated: ln_total_assets=0.286, ln_employees=0.617
Running model 'gva2'
✅ Model 'gva2' estimated: ln_total_assets=0.265, ln_employees=0.611
Running model 'gva3'
❌ Model 'gva3' failed. KeyError: 'gva3'
Model 'gva3' failed. Skipping.
Running model 'k1'


Traceback (most recent call last):
  File "c:\Users\lazyst\AppData\Local\python\pythoncore-3.14-64\Lib\site-packages\pandas\core\indexes\base.py", line 3641, in get_loc
    return self._engine.get_loc(casted_key)
           ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
  File "pandas/_libs/index.pyx", line 168, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/index.pyx", line 197, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/hashtable_class_helper.pxi", line 7668, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas/_libs/hashtable_class_helper.pxi", line 7676, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: 'gva3'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "C:\Users\lazyst\AppData\Local\Temp\ipykernel_28800\996074865.py", line 60, in run_panel
    f'ln_{p}': df_raw[p].apply(lambda x: np.log(x) if x > 0 else None) for p in params_named if p in mod.to_log
              

✅ Model 'k1' estimated: ln_fixed_total=0.065, ln_employees=0.720
Running model 'k2'
✅ Model 'k2' estimated: ln_tangibles=0.090, ln_intangibles=0.025, ln_investments_other=0.007, ln_employees=0.628
Panel regressions complete. 4 models, writing.


# 1a. LMM
$$
\begin{align*}
y_{it} &= \alpha_i + \gamma_t + w_{it}\theta + \beta E[TFP_{-i,g,t}\vert{}g] + \epsilon_{it} \\
x_{it} &=
\begin{pmatrix}
k_{it} & l_{it}
\end{pmatrix}
\end{align*}
$$
**Run 1**
- ✅ Model 'peer3' estimated: peer_tfp_ttwa_donut=0.033, peer_tfp_pc4_donut=0.015, peer_tfp_pc8=0.280
- ✅ Model 'peer3_no_firm_fe' estimated: peer_tfp_ttwa_donut=0.204, peer_tfp_pc4_donut=0.142, peer_tfp_pc8=0.559
- ✅ Model 'peer3_employees' estimated: ln_employees=0.009, peer_tfp_ttwa_donut=0.033, peer_tfp_pc4_donut=0.015, peer_tfp_pc8=0.280
Panel regressions complete. 3 models, writing.

In [2]:
panel_name = "working_yearly_g"
out_name = "results_1a_lmm_exog"

# 3. Models: varying Y (gva1 vs gva2) and K (fixed_total vs tangibles)
models = {
    'lmm_exog': {
        'Y': 'gva',
        'X': ['total_assets', 'employees', 'wg1_y', 'wg1_k', 'wg1_l'],
        'fe': ['i', 't'],
        'description': 'Strict exogeneity, structural form'
    },
    'lmm_instr': {
        'Y': 'gva',
        'X': ['total_assets', 'employees', 'wg1_y', 'wg1_k', 'wg1_l'],
        'Z': {
            'wg1_y': ['w2g1_k', 'w2g1_l'],
        },
        'fe': ['i', 't'],
        'description': 'Instrument endogenous effect, LMM'
    }
}

run_res_series: list[tuple[PanelEffectsResults, ModelSpec, str]] = []
worker_args = [
    (ModelSpec(**mod_obj), panel_name, m_name)
    for m_name, mod_obj in models.items() if ModelSpec(**mod_obj).include
]
# from concurrent.futures import ProcessPoolExecutor
# import numpy as np
# with ProcessPoolExecutor(max_workers=4) as executor:
#     panel_raw = executor.map(run_panel, worker_args)
#     i = 0
#     run_res_series.append(panel_raw) #type: ignore
#     # for panel_out in panel_raw:
#     #     if panel_out[0] is not None:
#     #         continue
#     #     res = panel_out[0]
#     #     mod = worker_args[i][0]
#     #     model_name = worker_args[i][2]
#     #     if res is None:
#     #         print(f"Model '{model_name}' failed. Skipping.")
#     #         continue
#     #     run_res_series.append((res, mod, model_name))
#     #     i += 1
for args in worker_args:
    mod, _, model_name = args
    res, beta, effects_dict = run_panel(args)
    if res is None:
        print(f"Model '{model_name}' failed. Skipping.")
        continue
    run_res_series.append((res, mod, model_name))   # type: ignore

model_count = len(run_res_series)
if model_count:
    output_str = "\n".join(map(format_str, run_res_series))
    print(f"Panel regressions complete. {model_count} model{'' if model_count == 1 else 's'}, writing.")
    with open(dirs.output_dir / f"{out_name}.txt", "w") as f:
        f.write(output_str)

Running model 'lmm_exog' as panel OLS
❌ Model 'lmm_exog' failed. ValueError: Series can only be used with a 2-level MultiIndex
Model 'lmm_exog' failed. Skipping.


Traceback (most recent call last):
  File "c:\Users\lazyst\Files\ucl\Dissertation\model\src\f_7_run_panel.py", line 86, in run_panel
    mod_panel = PanelOLS(Y, X, entity_effects='i' in mod.fe, time_effects='t' in mod.fe)
  File "c:\Users\lazyst\Files\ucl\Dissertation\.venv-main\Lib\site-packages\linearmodels\panel\model.py", line 1258, in __init__
    super().__init__(dependent, exog, weights=weights, check_rank=check_rank)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\lazyst\Files\ucl\Dissertation\.venv-main\Lib\site-packages\linearmodels\panel\model.py", line 315, in __init__
    self.dependent = PanelData(dependent, "Dep")
                     ~~~~~~~~~^^^^^^^^^^^^^^^^^^
  File "c:\Users\lazyst\Files\ucl\Dissertation\.venv-main\Lib\site-packages\linearmodels\panel\data.py", line 212, in __init__
    raise ValueError("Series can only be used with a 2-level MultiIndex")
ValueError: Series can only be used with a 2-level MultiIndex


Index(['registered_number', 'year', 'gva', 'total_assets', 'employees',
       'wg1_y', 'wg1_k', 'wg1_l', 'w2g1_k', 'w2g1_l'],
      dtype='str')
Running model 'lmm_instr' as panel IV, formula: gva ~ total_assets + employees + wg1_y + wg1_k + wg1_l | total_assets + employees + wg1_k + wg1_l + w2g1_k + w2g1_l


KeyboardInterrupt: 

# 2. Distance decay model

$$
\begin{align*}
z_{it} &= \alpha_i + \gamma_t + \rho \sum_{j \neq i} f(d_{ij}) \cdot z_{jt} + \epsilon_{it}   \\
y_{it} &= \alpha_i + \gamma_t + \beta_1 k_{it} + \beta_2 l_{it} + \rho \sum_{j \neq i} w_{ij} z_{jt} + \epsilon_{it}
\end{align*}
$$

In [ ]:
from linearmodels.panel.results import PanelEffectsResults
from f_7_run_panel import run_panel, ModelSpec, format_str

table_panel_name = "working_yearly_with_tfp_wave"

# 3. Models: varying Y (gva1 vs gva2) and K (fixed_total vs tangibles)
models = {
    'dd_gravity1': {
        'Y': 'tfp',
        'X': ['tfp_wav1'],
        'W': ['nb_peers', 'employees'],
        'to_log': ['nb_peers', 'employees'],
        'fe': ['i', 't'],
        'description': 'Base: 1/d distance peer effect'
    },
    'dd_neg2': {
        'Y': 'tfp',
        'X': ['tfp_wav2'],
        'W': ['nb_peers', 'employees'],
        'to_log': ['nb_peers', 'employees'],
        'fe': ['i', 't'],
        'description': 'Base: -1/d^2 distance peer effect'
    },
    'dd_expdd3': {
        'Y': 'tfp',
        'X': ['tfp_wav3'],
        'W': ['nb_peers', 'employees'],
        'to_log': ['nb_peers', 'employees'],
        'fe': ['i', 't'],
        'description': 'Base: exp(-d / 1000) distance peer effect'
    },
    'dd_struct1': {
        'Y': 'gva1',
        'X': ['tfp_wav1'],
        'W': ['total_assets', 'employees'],
        'to_log': ['gva1', 'total_assets', 'employees'],
        'fe': ['i', 't'],
        'description': 'Structural GVA model regressing with peer TFP'
    },
    'dd_struct2': {
        'Y': 'gva1',
        'X': ['tfp_wav2'],
        'W': ['total_assets', 'employees'],
        'to_log': ['gva1', 'total_assets', 'employees'],
        'fe': ['i', 't'],
        'description': 'Structural GVA model regressing with peer TFP'
    },
    'dd_struct3': {
        'Y': 'gva1',
        'X': ['tfp_wav3'],
        'W': ['total_assets', 'employees'],
        'to_log': ['gva1', 'total_assets', 'employees'],
        'fe': ['i', 't'],
        'description': 'Structural GVA model regressing with peer TFP'
    },
}

run_res_series: list[tuple[PanelEffectsResults, ModelSpec, str]] = []
worker_args = [
    (ModelSpec(**mod_obj), df_panel, m_name)
    for m_name, mod_obj in models.items() if ModelSpec(**mod_obj).include
]
for args in worker_args:
    mod, table_panel_name, model_name = args
    res, beta, effects_dict = run_panel(args)
    if res is None:
        print(f"Model '{model_name}' failed. Skipping.")
        continue
    run_res_series.append((res, mod, model_name))       # type: ignore

model_count = len(run_res_series)
if model_count:
    output_str = "\n".join(map(format_str, run_res_series))
    out_file = dirs.output_dir / "results_2_dd"
    print(f"Panel regressions complete. {model_count} model{'' if model_count == 1 else 's'}, writing to {out_file}")
    with open(f"{out_file}.txt", "w") as f:
        f.write(output_str)

Running model 'dd_gravity1'
✅ Model 'dd_gravity1' estimated: tfp_wav1=0.168, ln_nb_peers=0.013, ln_employees=-0.009
Running model 'dd_neg2'
✅ Model 'dd_neg2' estimated: tfp_wav2=0.135, ln_nb_peers=0.012, ln_employees=-0.009
Running model 'dd_expdd3'
✅ Model 'dd_expdd3' estimated: tfp_wav3=0.184, ln_nb_peers=0.016, ln_employees=-0.009
Running model 'dd_struct1'
✅ Model 'dd_struct1' estimated: tfp_wav1=0.168, ln_total_assets=0.265, ln_employees=0.619
Running model 'dd_struct2'
✅ Model 'dd_struct2' estimated: tfp_wav2=0.135, ln_total_assets=0.265, ln_employees=0.619
Running model 'dd_struct3'
✅ Model 'dd_struct3' estimated: tfp_wav3=0.183, ln_total_assets=0.265, ln_employees=0.619
Panel regressions complete. 6 models, writing to C:\Users\lazyst\Files\ucl\Dissertation\model\output\results_2_dd
